<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/01_RAG_Base_documental_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — RAG / Base documental

## ETAPA 0 — Diagnóstico del entorno

Este notebook construye una base documental RAG a partir de archivos PDF.

El flujo será:

PDF → texto → chunks → embeddings → FAISS → recuperación de contexto

En esta primera etapa se verifican:

- CPU
- RAM
- GPU disponible
- VRAM
- espacio en disco
- ruta del proyecto
- archivos PDF disponibles

El notebook funcionará tanto con CPU como con GPU.

In [4]:
# ============================================================
# ETAPA 0 - DIAGNÓSTICO DEL ENTORNO Y VALIDACIÓN
# ============================================================

from pathlib import Path
import os
import psutil
import torch

BASE_DIR = Path("/content/drive/MyDrive/Mx.Oscar.Hdez")

print("=" * 60)
print("DIAGNÓSTICO DEL ENTORNO")
print("=" * 60)

print("CPU:", os.cpu_count(), "núcleos")

ram = psutil.virtual_memory()
print(
    "RAM:",
    round(ram.total / 1024**3, 2), "GB total |",
    round(ram.available / 1024**3, 2), "GB disponible"
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
    DEVICE = "cuda"
else:
    print("GPU: No disponible")
    DEVICE = "cpu"

disk = psutil.disk_usage("/content")

print(
    "Disco:",
    round(disk.total / 1024**3, 2), "GB total |",
    round(disk.free / 1024**3, 2), "GB libres"
)

print()
print("=" * 60)
print("VALIDACIÓN DEL PROYECTO")
print("=" * 60)

pdf_files = sorted(BASE_DIR.glob("*.pdf"))

print("Ruta:", BASE_DIR)
print("Dispositivo:", DEVICE)
print("PDF encontrados:", len(pdf_files))

for pdf in pdf_files:
    print(" -", pdf.name)

DIAGNÓSTICO DEL ENTORNO
CPU: 2 núcleos
RAM: 12.67 GB total | 9.96 GB disponible
GPU: No disponible
Disco: 107.72 GB total | 87.06 GB libres

VALIDACIÓN DEL PROYECTO
Ruta: /content/drive/MyDrive/Mx.Oscar.Hdez
Dispositivo: cpu
PDF encontrados: 7
 - Manual_de_Lineamientos_TecNM.pdf
 - Perfil-Objetivo_ISC.pdf
 - REGLAMENTO_INTERIOR_2025.pdf
 - Reglamento_Estudiantes_TESOEM.pdf
 - Reglamento_Servicio_Social.pdf
 - Reglamento_de_uso_del_correo.pdf
 - Reticula_ISC.pdf


## ETAPA 1 — Instalación de dependencias

Se instalan únicamente las bibliotecas necesarias para construir el RAG:

- **PyMuPDF** para leer los archivos PDF.
- **Sentence Transformers** para generar embeddings.
- **FAISS** para crear y consultar el índice vectorial.

En este notebook no se instala todavía Ollama, Llama Stack, FastAPI ni WhatsApp.

In [5]:
# ============================================================
# ETAPA 1 - INSTALACIÓN DE DEPENDENCIAS
# ============================================================

!pip install -q \
    pymupdf \
    sentence-transformers \
    faiss-cpu

print("✓ Dependencias instaladas")

✓ Dependencias instaladas


## ETAPA 2 — Configuración del RAG y modelo de embeddings

En esta etapa se define:

- La carpeta donde se guardarán los resultados del RAG.
- El modelo de embeddings `multilingual-e5-small`.
- El tamaño de los chunks.
- El traslape entre chunks.
- La cantidad de resultados que recuperará FAISS.
- El dispositivo de ejecución: CPU o GPU.

Finalmente se carga el modelo E5 que convertirá los textos en vectores.

In [6]:
# ============================================================
# ETAPA 2 - CONFIGURACIÓN DEL RAG Y MODELO E5
# ============================================================

from sentence_transformers import SentenceTransformer

# Carpeta de salida
RAG_DIR = BASE_DIR / "rag"
RAG_DIR.mkdir(exist_ok=True)

# Archivos de salida
CHUNKS_FILE = RAG_DIR / "chunks.json"
INDEX_FILE = RAG_DIR / "faiss.index"

# Modelo de embeddings
EMBED_MODEL = "intfloat/multilingual-e5-small"

# Parámetros básicos
CHUNK_SIZE = 450
OVERLAP = 80
TOP_K = 4

print("Cargando modelo E5...")

embedding_model = SentenceTransformer(
    EMBED_MODEL,
    device=DEVICE
)

embedding_model.max_seq_length = 512
tokenizer = embedding_model.tokenizer

print("✓ Modelo E5 listo")
print("Dispositivo:", DEVICE)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("OVERLAP:", OVERLAP)
print("TOP_K:", TOP_K)
print("Salida:", RAG_DIR)

Cargando modelo E5...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✓ Modelo E5 listo
Dispositivo: cpu
CHUNK_SIZE: 450
OVERLAP: 80
TOP_K: 4
Salida: /content/drive/MyDrive/Mx.Oscar.Hdez/rag


## ETAPA 3 — Extracción y limpieza de los PDF

En esta etapa se leen los archivos PDF disponibles en la carpeta del proyecto.

Para cada página se realiza:

- extracción del texto;
- limpieza de espacios innecesarios;
- eliminación de saltos de línea excesivos;
- almacenamiento del nombre del documento, número de página y texto.

El resultado servirá como entrada para la creación de chunks.

In [8]:
# ============================================================
# ETAPA 3 - EXTRAER Y LIMPIAR TEXTO DE LOS PDF
# ============================================================

import pymupdf
import re

def limpiar_texto(texto):
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


documentos = []

for pdf_file in pdf_files:

    print("Procesando:", pdf_file.name)

    with pymupdf.open(pdf_file) as pdf:

        for pagina_num, pagina in enumerate(pdf, start=1):

            texto = limpiar_texto(
                pagina.get_text("text")
            )

            if texto:
                documentos.append({
                    "documento": pdf_file.name,
                    "pagina": pagina_num,
                    "texto": texto
                })

print()
print("✓ Extracción terminada")
print("PDF procesados:", len(pdf_files))
print("Páginas con texto:", len(documentos))

Procesando: Manual_de_Lineamientos_TecNM.pdf
Procesando: Perfil-Objetivo_ISC.pdf
Procesando: REGLAMENTO_INTERIOR_2025.pdf
Procesando: Reglamento_Estudiantes_TESOEM.pdf
Procesando: Reglamento_Servicio_Social.pdf
Procesando: Reglamento_de_uso_del_correo.pdf
Procesando: Reticula_ISC.pdf

✓ Extracción terminada
PDF procesados: 7
Páginas con texto: 378


## ETAPA 4 — Generación de chunks

En esta etapa el texto de cada página se divide en fragmentos o **chunks**.

Se utiliza el tokenizer del modelo E5 para trabajar por tokens y no por caracteres.

Cada chunk conserva:

- documento de origen;
- número de página;
- identificador del chunk;
- texto correspondiente.

También se utiliza un pequeño traslape entre chunks para evitar perder contexto entre fragmentos consecutivos.

In [9]:
# ============================================================
# ETAPA 4 - GENERAR CHUNKS
# ============================================================

chunks = []
chunk_id = 0

paso = CHUNK_SIZE - OVERLAP

for documento in documentos:

    tokens = tokenizer(
        documento["texto"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    for inicio in range(0, len(tokens), paso):

        final = min(
            inicio + CHUNK_SIZE,
            len(tokens)
        )

        fragmento = tokenizer.decode(
            tokens[inicio:final],
            skip_special_tokens=True
        ).strip()

        if fragmento:

            chunks.append({
                "chunk_id": chunk_id,
                "documento": documento["documento"],
                "pagina": documento["pagina"],
                "texto": fragmento
            })

            chunk_id += 1

        if final >= len(tokens):
            break

print("✓ Chunks generados:", len(chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors


✓ Chunks generados: 574


## ETAPA 5 — Embeddings e índice FAISS

En esta etapa cada chunk se convierte en un vector mediante el modelo E5.

Para trabajar correctamente con E5, cada fragmento se envía con el prefijo:

`passage:`

Después los vectores se normalizan y se almacenan en un índice FAISS.

Este índice permitirá buscar rápidamente los fragmentos más similares a una pregunta.

In [10]:
# ============================================================
# ETAPA 5 - GENERAR EMBEDDINGS Y CREAR FAISS
# ============================================================

import faiss
import numpy as np

textos = [
    "passage: " + chunk["texto"]
    for chunk in chunks
]

print("Generando embeddings...")

vectores = embedding_model.encode(
    textos,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True
)

vectores = np.asarray(
    vectores,
    dtype="float32"
)

index = faiss.IndexFlatIP(
    vectores.shape[1]
)

index.add(vectores)

print()
print("✓ Índice FAISS creado")
print("Chunks:", len(chunks))
print("Vectores:", index.ntotal)
print("Dimensiones:", vectores.shape[1])

Generando embeddings...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]


✓ Índice FAISS creado
Chunks: 574
Vectores: 574
Dimensiones: 384


## ETAPA 6 — Guardar la base RAG

En esta etapa se guardan en Google Drive:

- Los chunks con su documento, página y texto.
- El índice vectorial FAISS.

Estos archivos permitirán reutilizar el RAG posteriormente sin volver a procesar todos los PDF.

In [11]:
# ============================================================
# ETAPA 6 - GUARDAR CHUNKS E ÍNDICE FAISS
# ============================================================

import json

with open(
    CHUNKS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

faiss.write_index(
    index,
    str(INDEX_FILE)
)

print("✓ RAG guardado")
print("Chunks:", CHUNKS_FILE)
print("Índice:", INDEX_FILE)

✓ RAG guardado
Chunks: /content/drive/MyDrive/Mx.Oscar.Hdez/rag/chunks.json
Índice: /content/drive/MyDrive/Mx.Oscar.Hdez/rag/faiss.index


## ETAPA 7 — Prueba de recuperación

En esta etapa se prueba el funcionamiento del índice FAISS.

El proceso es:

Pregunta → embedding E5 → búsqueda FAISS → chunks más similares

Para las preguntas, E5 utiliza el prefijo:

`query:`

Se mostrarán los fragmentos recuperados junto con:

- documento;
- página;
- score de similitud;
- texto recuperado.

Esta prueba permite validar el RAG antes de conectarlo con el modelo generativo.

In [12]:
# ============================================================
# ETAPA 7 - PRUEBA DE RECUPERACIÓN
# ============================================================

def buscar(pregunta, top_k=TOP_K):

    query = "query: " + pregunta

    vector = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )

    vector = np.asarray(
        vector,
        dtype="float32"
    )

    scores, indices = index.search(
        vector,
        top_k
    )

    print("=" * 60)
    print("PREGUNTA:")
    print(pregunta)
    print("=" * 60)

    for n, (score, posicion) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        chunk = chunks[posicion]

        print()
        print(
            f"[{n}] {chunk['documento']} "
            f"| pág. {chunk['pagina']} "
            f"| score {score:.4f}"
        )

        print(chunk["texto"])
        print("-" * 60)

In [13]:
buscar(
    "¿Cuál es el perfil de egreso de Ingeniería en Sistemas Computacionales?"
)

PREGUNTA:
¿Cuál es el perfil de egreso de Ingeniería en Sistemas Computacionales?

[1] Perfil-Objetivo_ISC.pdf | pág. 1 | score 0.8601
©TecNM mayo 2016 Página | 1 TECNOLÓGICO NACIONAL DE MÉXICO Secretaría Académica, de Investigación e Innovación Dirección de Docencia e Innovación Educativa Ingeniería en Sistemas Computacionales ISIC-2010-224 Objetivo General Formar profesionistas líderes con visión estratégica y amplio sentido ético; capaz de diseñar, desarrollar, implementar y administrar tecnología computacional para aportar soluciones innovadoras en beneficio de la sociedad; en un contexto global, multidisciplinario y sostenible. Perfil de Egreso 1. Implementa aplicaciones computacionales para solucionar problemas de diversos contextos, integrando diferentes tecnologías, plataformas o dispositivos. 2. Diseña, desarrolla y aplica modelos computacionales para solucionar problemas, mediante la selección y uso de herramientas matemáticas. 3. Diseña e implementa interfaces para la automa

## ETAPA 8 — Validación final

En esta etapa se verifica que la base RAG guardada pueda reutilizarse.

Se cargan nuevamente:

- `chunks.json`
- `faiss.index`

Después se comprueba que el número de chunks coincida con el número de vectores almacenados en FAISS.

Con esto queda validado que el RAG puede ser utilizado posteriormente desde otro notebook.

In [14]:
# ============================================================
# ETAPA 8 - VALIDACIÓN FINAL DEL RAG
# ============================================================

import json
import faiss

with open(
    CHUNKS_FILE,
    "r",
    encoding="utf-8"
) as f:

    chunks_guardados = json.load(f)

index_guardado = faiss.read_index(
    str(INDEX_FILE)
)

print("=" * 60)
print("VALIDACIÓN FINAL DEL RAG")
print("=" * 60)

print("Chunks guardados:", len(chunks_guardados))
print("Vectores FAISS:", index_guardado.ntotal)

if len(chunks_guardados) == index_guardado.ntotal:
    print("✓ RAG válido y listo para reutilizar")
else:
    print("⚠ Existe una diferencia entre chunks y vectores")

VALIDACIÓN FINAL DEL RAG
Chunks guardados: 574
Vectores FAISS: 574
✓ RAG válido y listo para reutilizar
